**Aim**

To build a data pipeline that extracts data from a PostgreSQL table, transforms it, and loads the result into a new target table within the same database.

**Theory**

A data pipeline is a sequence of operations used to move and process data from a source to a destination. In this experiment, both the source and destination are PostgreSQL tables within the same database.

The first stage is Extract, where data is retrieved from an existing PostgreSQL source table using a database connection and SQL query. The extracted records can be loaded into a Pandas DataFrame for processing.

The second stage is Transform, where the extracted data is modified into the required format. Transformation may include selecting required columns, renaming columns, standardizing values, removing unnecessary records, or performing calculations.

The final stage is Load, where the transformed data is inserted into a new target table in the same PostgreSQL database. The target table stores the processed result separately from the original source table.

In [ ]:
!apt-get update -qq
!apt-get install -y postgresql postgresql-contrib -qq
!service postgresql start


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package libjson-perl.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../00-libjson-perl_4.10000-1_all.deb ...
Unpacking libjson-perl (4.10000-1) ...
Selecting previously unselected package postgresql-client-common.
Preparing to unpack .../01-postgresql-client-common_257build1.1_all.deb ...
Unpacking postgresql-client-common (257build1.1) ...
Selecting previously unselected package ssl-cert.
Preparing to unpack .../02-ssl-cert_1.1.2ubuntu1_all.deb ...
Unpacking ssl-cert (1.1.2ubuntu1) ...
Selecting previously unselected package postgresql-common.
Preparing to unpack .../03-postgresql-common_257build1.1_all.deb ...
Adding 'diversion of /usr/bin/pg_config to /usr/bin/pg_config.libpq-dev by postgres

In [ ]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres psql -c "CREATE DATABASE pipeline_db;"

ALTER ROLE
CREATE DATABASE


In [ ]:
pip install -q psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 56.9 MB/s eta 0:00:00


In [ ]:
import psycopg2

conn = psycopg2.connect(
    dbname="pipeline_db",
    user="postgres",
    password="postgres",
    host="localhost",
    port="5432"
)

cur = conn.cursor()

print("Connected to PostgreSQL successfully!")

Connected to PostgreSQL successfully!


In [ ]:
cur.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id SERIAL PRIMARY KEY,
    name VARCHAR(50),
    department VARCHAR(50),
    salary INTEGER
)
""")

cur.execute("DELETE FROM employees")

data = [
    ("Rahul", "IT", 50000),
    ("Priya", "HR", 35000),
    ("Amit", "IT", 60000),
    ("Sneha", "Finance", 28000),
    ("Rohan", "IT", 45000)
]

cur.executemany(
    "INSERT INTO employees (name, department, salary) VALUES (%s, %s, %s)",
    data
)

conn.commit()

print("Source data inserted successfully!")

Source data inserted successfully!


In [ ]:
cur.execute("""
SELECT name, department, salary
FROM employees
WHERE salary > 40000
""")

rows = cur.fetchall()

for row in rows:
    print(row)

('Rahul', 'IT', 50000)
('Amit', 'IT', 60000)
('Rohan', 'IT', 45000)


In [ ]:
cur.execute("DROP TABLE IF EXISTS high_salary_employees")

cur.execute("""
CREATE TABLE high_salary_employees AS
SELECT name, department, salary
FROM employees
WHERE salary > 40000
""")

conn.commit()

print("Transformed data loaded into target table!")

Transformed data loaded into target table!


In [ ]:
cur.execute("SELECT * FROM high_salary_employees")

rows = cur.fetchall()

print("Target Table:")
for row in rows:
    print(row)

Target Table:
('Rahul', 'IT', 50000)
('Amit', 'IT', 60000)
('Rohan', 'IT', 45000)


In [ ]:
cur.close()
conn.close()

print("PostgreSQL connection closed.")

PostgreSQL connection closed.


**Conclusion**

The SQL Data Pipeline was successfully implemented in Google Colab using Python and PostgreSQL. Data was extracted from the source employees table and transformed by filtering employees according to their salary. The transformed records were then loaded into a new high_salary_employees target table within the same database and verified successfully.